# TP Jour 1 — Choix du Sujet Kaggle, Stockage Distribué HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et votre jeu de données public sur **Kaggle**.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner un dataset Kaggle adapté au calcul distribué (~50 à 500 Mo).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts sur HDFS** et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** et inspecter votre premier job sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Titre du sujet :** `• Analyse distribuée et segmentation comportementale de transactions financières via une architecture Data Lakehouse (HDFS, PySpark, Delta Lake & Streamlit)`

- **Identifiant Kaggle (`YAO MARCEL/transactions_data`) :** Fichier principal: `transactions_data.csv`

- **Problématique métier principale :** `Comment exploiter une infrastructure Big Data tolérante aux pannes pour analyser la concentration sectorielle des flux monétaires (loi de Pareto), mesurer l'impact du canal de paiement sur les incidents techniques et segmenter les porteurs de cartes selon leurs comportements d'achat ?`

- **Variables clés retenues (temporelles, quantitatives, catégorielles) :**

• **Temporelles :** date (horodatage complet), dérivées en tx_year_month (granularité mensuelle) et tx_hour (tranche horaire).

• **Quantitatives :** amount (convertie en amount_usd décimal), complétée par des métriques agrégées (frequence_tx, volume_total_usd, panier_median_usd).

• **Catégorielles & Identifiants :** use_chip (mode de paiement : puce, piste, en ligne), mcc (code catégorie marchand), merchant_state (localisation géographique), errors / has_error (indicateur d'anomalie) et client_id (identifiant client pour le clustering).

## 1. Téléchargement de votre Dataset Kaggle dans `/workspace/data_raw`

Renseignez dans la cellule suivante l'identifiant Kaggle de votre dataset (ex. `"auteur/mon-dataset"`), ou déposez directement vos fichiers dans le dossier `data_raw/` à la racine du projet.
> **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

lien du Dataset: https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets/data

In [1]:
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# TODO : Renseignez l'identifiant Kaggle de votre sujet (format "auteur/nom-du-dataset")
KAGGLE_DATASET = "computingvictor/transactions-fraud-datasets"
  # Exemple : "auteur/nom-du-dataset"

local_files = sorted([f for f in DATA_RAW.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

if not local_files:
    assert KAGGLE_DATASET.strip(), (
        "TODO : Renseignez `KAGGLE_DATASET = 'auteur/nom-du-dataset'` (ou déposez vos fichiers dans data_raw/)."
    )
    import kagglehub
    print(f"Téléchargement de {KAGGLE_DATASET} depuis Kaggle Hub...")
    cache_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    for src_file in sorted(cache_dir.rglob("*")):
        if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
            shutil.copy2(src_file, DATA_RAW / src_file.name)
    local_files = sorted([f for f in DATA_RAW.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, "Aucun fichier de données détecté dans /workspace/data_raw/."

print(f"Inventaire local dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {f.name:<45} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Inventaire local dans /workspace/data_raw (5 fichier(s)) :
 - cards_data.csv                                   0.49 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - mcc_codes.json                                   0.00 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - train_fraud_labels.json                        151.71 Mo  (~10 bloc(s) HDFS de 16 Mo)
 - transactions_data.csv                         1200.23 Mo  (~76 bloc(s) HDFS de 16 Mo)
 - users_data.csv                                   0.16 Mo  (~1 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialiser la SparkSession
spark = SparkSession.builder.appName("J1_HDFS_Ingestion_Bronze").getOrCreate()

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# 2. Lire les 3 paramètres HDFS depuis `spark.conf.get(...)`
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1729329b-c35f-4581-a5c2-f558b0508dc7;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 520ms :: artifacts dl 23ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

Spark version       : 3.5.3
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{file_path.name}")
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/cards_data.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/mcc_codes.json
Téléversé vers HDFS : /lakehouse/bronze/raw/train_fraud_labels.json
Téléversé vers HDFS : /lakehouse/bronze/raw/transactions_data.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/users_data.csv


In [4]:
# Inspection physique du découpage en blocs de 16 Mo et identification du plus gros fichier pour le crash-test
statuses = sorted(fs.listStatus(raw_hdfs_dir), key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {path.getName()} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : transactions_data.csv (1200.23 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #7 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #8 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #9 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #10 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #11 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #12 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #13 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #14 (16.00 Mo) -> DataNodes : ['datano

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

---

Réponses aux questions d'analyse (Section 3)


1. Comparaison du nombre de blocs sur http://localhost:9870 :
	• le fichier fait 1200,23 Mo. En divisant par 16 Mo (\(1200,23 / 16 = 75,01\)), HDFS le découpe physiquement en 76 blocs (75 blocs pleins de 16 Mo + 1 dernier bloc contenant le reste, soit ~0,23 Mo).
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo sur le disque ?
	• Non. Contrairement à un système de fichiers classique, la taille de bloc HDFS (16 Mo) est uniquement une limite maximale de découpage logique. Un fichier de 3 Mo n'occupe que 3 Mo réels sur le disque dur des DataNodes.
3. Pourquoi configurer dfs.replication = 2 sur un cluster de 3 DataNodes ?
	• Cela assure la tolérance aux pannes sans gaspiller d'espace : chaque bloc est copié sur 2 machines différentes. Si un DataNode tombe en panne, la donnée reste disponible sur le second, et le 3e DataNode sert de serveur de secours pour recréer immédiatement une nouvelle copie.

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !



### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [5]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# Lecture du fichier CSV depuis HDFS pendant la panne
df_check = spark.read.csv(largest_hdfs_path, header=True, inferSchema=True)

assert df_check is not None, "TODO : Lisez `largest_hdfs_path` dans `df_check`."

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")


Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/transactions_data.csv


Nombre total de lignes lues       : 13,305,915
Nombre de partitions Spark créées : 16


Vous avons obtenu 16 partitions Spark (et non 76) 

• Pourquoi ? Par défaut, Spark limite le parallélisme de lecture de fichiers texte/CSV au nombre de cœurs CPU alloués au conteneur Docker (ou regroupe plusieurs blocs de 16 Mo dans une même partition via spark.sql.files.maxPartitionBytes fixé par défaut à 128 Mo, soit \(1200\text{ Mo} / 128\text{ Mo} \approx 10\) à \(16\) partitions selon vos cœurs CPU).

• Le HDFS a bien découpé le fichier en 76 blocs de 16 Mo ( avec 49 + 48 + 55 = 152 blocs répliqués par 2, soit \(152 / 2 = 76\) blocs uniques), et que Spark a lu les blocs en parallèle sur 16 partitions en s'appuyant sur les réplicas survivants de datanode1 et datanode3 grâce à la Data Locality.


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts en tables Delta Lake Bronze
Dans une architecture **Médaillon**, la couche **Bronze** conserve les données brutes fidèles à la source tout en les stockant au format transactionnel **Delta Lake** enrichi de métadonnées de traçabilité :
- `_ingested_at` : `F.current_timestamp()`
- `_source_file` : `F.input_file_name()`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de votre sujet dans `/lakehouse/bronze/<nom_table>`.

In [6]:
# 1. Définition de la source brute
BRONZE_SOURCES: dict[str, str] = {
    "transactions": "/lakehouse/bronze/raw/transactions_data.csv",
}

assert len(BRONZE_SOURCES) > 0, "TODO : Renseignez au moins une table à ingérer dans `BRONZE_SOURCES`."

for table_name, raw_path in BRONZE_SOURCES.items():
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    # 2. Lecture du fichier CSV brut depuis HDFS
    df_raw = spark.read.csv(raw_path, header=True, inferSchema=True)

    # 3. Normalisation des noms de colonnes en snake_case (sans espaces ni majuscules)
    clean_cols = [
        c.strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_") for c in df_raw.columns
    ]
    df_clean = df_raw.toDF(*clean_cols)

    # 4. Ajout des métadonnées de traçabilité (audit)
    df_bronze = df_clean.withColumn("_ingested_at", F.current_timestamp()).withColumn(
        "_source_file", F.input_file_name()
    )

    # 5. Écriture au format transactionnel Delta Lake sur HDFS
    (
        df_bronze.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )

    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<38} -> {cnt:>10,} lignes")


26/11/26 11:56:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/26 11:57:01 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers


[BRONZE OK] /lakehouse/bronze/transactions         -> 13,305,915 lignes


In [8]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['transactions']
